# Analysis of neighboring pixels

This notebook is for investigating how model performance changes if we extend the input features to include neighboring pixels as well as the one we are trying to predict. My hope is that these models will perform significantly better than the previous models I have made.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset, TensorDataset
from scipy.spatial import cKDTree
from sklearn.metrics import roc_auc_score

device = torch.device("cuda")
print(f"Using: {device}")

EMBEDDING_COLS = [f'A{i:02d}' for i in range(64)]

Using: cuda


In [2]:
def build_neighbour_table(df, patch_size=3, tolerance=0.6):
    """
    Pre-computes a (N, patch_size, patch_size) integer array of neighbour indices.
    Entry is -1 where no neighbour exists (boundary pixels, zero-padded later).
    
    tolerance: fraction of pixel spacing within which we accept a candidate 
               as a true neighbour (0.6 means within 60% of one pixel width).
               This handles slight coordinate irregularities from projected exports.
    """
    half = patch_size // 2
    coords = np.stack([df['lon'].values, df['lat'].values], axis=1)  # (N, 2)
    
    # Estimate pixel spacing (used only for tolerance threshold, not for lookup)
    pixel_lon = np.median(np.diff(np.sort(df['lon'].unique())))
    pixel_lat = np.median(np.diff(np.sort(df['lat'].unique())))
    
    print(f"Building KD-tree for {len(df):,} pixels...")
    tree = cKDTree(coords)
    
    # k = patch_size^2 gives us enough candidates to fill the whole patch
    k = patch_size ** 2 + 1  # +1 to include self
    print(f"Querying {k} nearest neighbours per pixel (this may take a few minutes)...")
    distances, indices = tree.query(coords, k=k, workers=-1)  # workers=-1 = all CPUs
    
    neighbour_table = np.full((len(df), patch_size, patch_size), -1, dtype=np.int32)
    
    for slot in range(k):
        neighbour_coords = coords[indices[:, slot]]           # (N, 2)
        delta_lon = neighbour_coords[:, 0] - coords[:, 0]    # (N,)
        delta_lat = neighbour_coords[:, 1] - coords[:, 1]    # (N,)
        
        # Convert offsets to grid units
        dc = np.round(delta_lon / pixel_lon).astype(int)     # col offset
        dr = np.round(delta_lat / pixel_lat).astype(int)     # row offset
        
        # Check offset is within the patch and within tolerance of a true grid position
        residual_lon = np.abs(delta_lon - dc * pixel_lon)
        residual_lat = np.abs(delta_lat - dr * pixel_lat)
        
        valid = (
            (np.abs(dc) <= half) &
            (np.abs(dr) <= half) &
            (residual_lon < tolerance * pixel_lon) &
            (residual_lat < tolerance * pixel_lat)
        )
        
        patch_row = dr + half
        patch_col = dc + half

        # Clip to valid range before indexing — out-of-bounds slots are excluded
        # by the `valid` mask, but numpy evaluates the index expression first.
        patch_row_safe = np.clip(patch_row, 0, patch_size - 1)
        patch_col_safe = np.clip(patch_col, 0, patch_size - 1)

        pixel_indices = np.arange(len(df))
        mask = valid & (neighbour_table[pixel_indices, patch_row_safe, patch_col_safe] == -1)
        neighbour_table[pixel_indices[mask], patch_row_safe[mask], patch_col_safe[mask]] = \
            indices[mask, slot]
    
    # Diagnostics
    surrounding = neighbour_table.reshape(len(df), patch_size * patch_size)
    centre_idx = half * patch_size + half
    mask_no_centre = np.ones(patch_size * patch_size, dtype=bool)
    mask_no_centre[centre_idx] = False
    all_8_neighbours = np.all(surrounding[:, mask_no_centre] != -1, axis=1)
    zero_neighbours  = np.all(surrounding[:, mask_no_centre] == -1, axis=1)
    
    print(f"\nNeighbour table built:")
    print(f"  Pixels with all 8 neighbours: {all_8_neighbours.mean()*100:.1f}%  "
          f"(expected ~82%)")
    print(f"  Pixels with 0 neighbours:     {zero_neighbours.mean()*100:.1f}%")
    print(f"  Mean neighbours per pixel:    "
          f"{(surrounding[:, mask_no_centre] != -1).sum(axis=1).mean():.2f}")
    
    return neighbour_table, pixel_lon, pixel_lat


class GlacierPatchDataset(Dataset):
    """
    Serves spatial patches of AlphaEarth embeddings centred on each pixel.
    Uses a pre-computed KD-tree neighbour table for robust, fast lookup.
    
    Input tensor shape:  (64, patch_size, patch_size)
    Target:              binary melt_label (float32)
    """
    def __init__(self, parquet_path, patch_size=3, tolerance=0.6):
        assert patch_size % 2 == 1, "patch_size must be odd"
        self.patch_size = patch_size
        self.half = patch_size // 2

        print(f"Loading {parquet_path} ...")
        df = pd.read_parquet(parquet_path)
        print(f"  {len(df):,} pixels")

        self.embeddings = df[EMBEDDING_COLS].values.astype(np.float32)  # (N, 64)
        self.labels     = df['melt_label'].values.astype(np.float32)

        self.neighbour_table, _, _ = build_neighbour_table(df, patch_size, tolerance)
        # neighbour_table: (N, patch_size, patch_size) int32, -1 = missing

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        slots = self.neighbour_table[idx]          # (3, 3)
        flat  = slots.flatten()                    # (9,)
        
        # Replace -1 (missing) with 0 temporarily for indexing
        valid = flat.copy()
        missing = valid == -1
        valid[missing] = 0
        
        # Fetch all 9 embeddings at once — one array index operation
        patch = self.embeddings[valid]             # (9, 64)
        
        # Zero out missing neighbours
        patch[missing] = 0.0
        
        # Reshape to (64, 3, 3)
        patch = patch.T.reshape(64, 3, 3)
        
        return torch.from_numpy(patch.copy()), torch.tensor(self.labels[idx])

In [3]:
PARQUET_DIR = "/nvme1/users/md962/glacier/Glacier Project/Merged/"  

REGIONS = ["r1a", "r1b", "r2", "r3"]

datasets = {}
for region in REGIONS:
    print(f"\n--- Loading {region} ---")
    datasets[region] = GlacierPatchDataset(
        parquet_path=f"{PARQUET_DIR}/{region}_combined.parquet",
        patch_size=3
    )


--- Loading r1a ---
Loading /nvme1/users/md962/glacier/Glacier Project/Merged//r1a_combined.parquet ...
  6,222,913 pixels
Building KD-tree for 6,222,913 pixels...
Querying 10 nearest neighbours per pixel (this may take a few minutes)...

Neighbour table built:
  Pixels with all 8 neighbours: 92.6%  (expected ~82%)
  Pixels with 0 neighbours:     0.0%
  Mean neighbours per pixel:    7.81

--- Loading r1b ---
Loading /nvme1/users/md962/glacier/Glacier Project/Merged//r1b_combined.parquet ...
  1,028,089 pixels
Building KD-tree for 1,028,089 pixels...
Querying 10 nearest neighbours per pixel (this may take a few minutes)...

Neighbour table built:
  Pixels with all 8 neighbours: 84.3%  (expected ~82%)
  Pixels with 0 neighbours:     0.0%
  Mean neighbours per pixel:    7.57

--- Loading r2 ---
Loading /nvme1/users/md962/glacier/Glacier Project/Merged//r2_combined.parquet ...
  6,217,318 pixels
Building KD-tree for 6,217,318 pixels...
Querying 10 nearest neighbours per pixel (this may ta

In [ ]:
def build_gpu_tensors(dataset):
    """
    Precomputes all patches and loads them directly onto the GPU as tensors.
    __getitem__ is only called during this build step, never during training.
    """
    n = len(dataset)
    print(f"Precomputing {n:,} patches...")
    
    # Preallocate CPU arrays
    all_patches = np.zeros((n, 64, 3, 3), dtype=np.float32)
    all_labels  = np.zeros(n, dtype=np.float32)
    
    # Fill using vectorised __getitem__
    batch_size = 100000  # process in chunks to show progress
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        for i in range(start, end):
            patch, label = dataset[i]
            all_patches[i] = patch.numpy()
            all_labels[i]  = label.item()
        print(f"  {end:,} / {n:,} done", end='\r')
    
    print(f"\nMoving to GPU...")
    patches_gpu = torch.from_numpy(all_patches).cuda()
    labels_gpu  = torch.from_numpy(all_labels).cuda()
    
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    return TensorDataset(patches_gpu, labels_gpu)

# Build for each region
print("=== r1a ===")
train_tensors_r1a = build_gpu_tensors(datasets["r1a"])
print("=== r1b ===")
train_tensors_r1b = build_gpu_tensors(datasets["r1b"])
print("=== r2 ===")
train_tensors_r2  = build_gpu_tensors(datasets["r2"])
print("=== r3 (val) ===")
val_tensors       = build_gpu_tensors(datasets["r3"])

# Combine training regions
train_tensors = ConcatDataset([train_tensors_r1a, train_tensors_r1b, train_tensors_r2])
print(f"\nTotal training pixels: {len(train_tensors):,}")
print(f"Total val pixels:      {len(val_tensors):,}")

=== r1a ===
Precomputing 6,222,913 patches...
  6,222,913 / 6,222,913 done
Moving to GPU...
GPU memory used: 14.5 GB
=== r1b ===
Precomputing 1,028,089 patches...
  1,028,089 / 1,028,089 done
Moving to GPU...
GPU memory used: 16.9 GB
=== r2 ===
Precomputing 6,217,318 patches...
  6,217,318 / 6,217,318 done
Moving to GPU...
GPU memory used: 31.2 GB
=== r3 (val) ===
Precomputing 1,774,319 patches...
  1,774,319 / 1,774,319 done
Moving to GPU...
GPU memory used: 35.3 GB

Total training pixels: 13,468,320
Total val pixels:      1,774,319


In [5]:
# Skip the build step — load from disk
train_tensors_r1a = torch.load('/nvme1/users/md962/glacier/tensors_r1a.pt', weights_only=False)
train_tensors_r1b = torch.load('/nvme1/users/md962/glacier/tensors_r1b.pt', weights_only=False)
train_tensors_r2  = torch.load('/nvme1/users/md962/glacier/tensors_r2.pt', weights_only=False)
val_tensors       = torch.load('/nvme1/users/md962/glacier/tensors_r3.pt', weights_only=False)

train_tensors = ConcatDataset([train_tensors_r1a, train_tensors_r1b, train_tensors_r2])

In [7]:
class GPUDataLoader:
    """
    Replaces DataLoader entirely for data already on GPU.
    Shuffling and batching done directly on GPU — no CPU involvement.
    """
    def __init__(self, tensor_dataset, batch_size, shuffle=True):
        # Extract tensors directly from TensorDataset
        self.patches    = tensor_dataset.tensors[0]  # (N, 64, 3, 3) on GPU
        self.labels     = tensor_dataset.tensors[1]  # (N,) on GPU
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.n          = len(self.labels)
    
    def __len__(self):
        return (self.n + self.batch_size - 1) // self.batch_size
    
    def __iter__(self):
        if self.shuffle:
            idx = torch.randperm(self.n, device='cuda')
        else:
            idx = torch.arange(self.n, device='cuda')
        
        for start in range(0, self.n, self.batch_size):
            batch_idx = idx[start:start + self.batch_size]
            yield self.patches[batch_idx], self.labels[batch_idx]

In [8]:
# Concatenate all training regions into single tensors
train_patches = torch.cat([
    train_tensors_r1a.tensors[0],
    train_tensors_r1b.tensors[0],
    train_tensors_r2.tensors[0]
], dim=0)

train_labels = torch.cat([
    train_tensors_r1a.tensors[1],
    train_tensors_r1b.tensors[1],
    train_tensors_r2.tensors[1]
], dim=0)

train_combined = TensorDataset(train_patches, train_labels)

print(f"Combined training tensor: {train_patches.shape}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Combined training tensor: torch.Size([13468320, 64, 3, 3])
GPU memory used: 66.3 GB


In [9]:
train_loader = GPUDataLoader(train_combined, batch_size=65536, shuffle=True)
val_loader   = GPUDataLoader(val_tensors,    batch_size=65536, shuffle=False)

In [10]:
class GlacierCNN(nn.Module):
    """
    Patch-based CNN for per-pixel glacier melt prediction.
    
    Input:  (B, 64, 3, 3) — 64-channel AlphaEarth embedding patch
    Output: (B, 1)        — raw logit (apply sigmoid for probability)
    
    Architecture: two Conv2d layers with 2x2 kernels progressively 
    reduce the 3x3 spatial dims to 1x1, then FC layers classify.
    We use 2x2 kernels (not 3x3) so we get two conv stages rather 
    than collapsing to 1x1 in one step — this lets the network learn
    hierarchical spatial features rather than just a weighted sum.
    """
    def __init__(self, dropout=0.3):
        super().__init__()
        
        self.conv_block = nn.Sequential(
            # (B, 64, 3, 3) -> (B, 128, 2, 2)
            nn.Conv2d(64, 128, kernel_size=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # (B, 128, 2, 2) -> (B, 256, 1, 1)
            nn.Conv2d(128, 256, kernel_size=2),
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        
        self.fc_block = nn.Sequential(
            nn.Flatten(),                        # (B, 256)
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)                     # raw logit
        )
    
    def forward(self, x):
        return self.fc_block(self.conv_block(x)).squeeze(1)

# Quick architecture check
model = GlacierCNN()
dummy = torch.zeros(8, 64, 3, 3)
out = model(dummy)
print(f"Output shape: {out.shape}")     # should be torch.Size([8])
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Output shape: torch.Size([8])
Trainable parameters: 202,049


In [12]:
# Recompute pos_weight from loaded tensors
all_labels = torch.cat([
    train_tensors_r1a.tensors[1],
    train_tensors_r1b.tensors[1],
    train_tensors_r2.tensors[1]
]).cpu().numpy()

melt_rate  = all_labels.mean()
pos_weight = torch.tensor((1 - melt_rate) / melt_rate, dtype=torch.float32)
print(f"Training melt rate: {melt_rate:.3f}")
print(f"pos_weight: {pos_weight:.2f}")

Training melt rate: 0.189
pos_weight: 4.29


In [14]:
device = torch.device("cuda")
print(f"Using device: {device}")

model = GlacierCNN(dropout=0.5).to(device)  # increased from 0.3
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)  # added weight decay
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)
# Scheduler reduces LR by half if val AUC stops improving for 2 epochs

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    all_labels, all_probs = [], []
    
    with torch.set_grad_enabled(train):
        for patches, labels in loader:
            patches = patches.to(device)
            labels  = labels.to(device)
            
            logits = model(patches)
            loss   = criterion(logits, labels)
            
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            total_loss += loss.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().detach().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / loader.n
    auc      = roc_auc_score(all_labels, all_probs)
    # AUC-ROC is the right metric here: it's threshold-independent and 
    # handles class imbalance properly, unlike raw accuracy
    return avg_loss, auc

EARLY_STOPPING_PATIENCE = 5

N_EPOCHS = 20
best_val_auc = 0
best_epoch   = 0

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train AUC':>9}  {'Val Loss':>8}  {'Val AUC':>7}")
print("-" * 55)

for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_auc = run_epoch(train_loader, train=True)
    val_loss,   val_auc   = run_epoch(val_loader,   train=False)
    
    scheduler.step(val_auc)
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch   = epoch
        torch.save(model.state_dict(), "best_glacier_cnn.pt")
    
    print(f"{epoch:>5}  {train_loss:>10.4f}  {train_auc:>9.4f}  "
          f"{val_loss:>8.4f}  {val_auc:>7.4f}"
          + (" ← best" if epoch == best_epoch else "")
          + (f"  [LR → {optimizer.param_groups[0]['lr']:.2e}]" 
             if optimizer.param_groups[0]['lr'] < 1e-3 else ""))
    
    # Early stopping
    if epoch - best_epoch >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping — no improvement for {EARLY_STOPPING_PATIENCE} epochs")
        break

print(f"\nBest val AUC: {best_val_auc:.4f} at epoch {best_epoch}")

Using device: cuda
Epoch  Train Loss  Train AUC  Val Loss  Val AUC
-------------------------------------------------------
    1      0.4416     0.9545    0.7204   0.9348 ← best
    2      0.3590     0.9687    0.9725   0.9432 ← best
    3      0.3340     0.9725    0.9968   0.9456 ← best
    4      0.3176     0.9749    0.8587   0.9447
    5      0.3055     0.9765    1.3300   0.9478 ← best
    6      0.2963     0.9778    1.1617   0.9474
    7      0.2894     0.9787    1.2265   0.9463
    8      0.2838     0.9795    1.1480   0.9489 ← best
    9      0.2794     0.9800    1.1184   0.9486
   10      0.2737     0.9808    1.1143   0.9418
   11      0.2714     0.9811    1.3131   0.9445  [LR → 5.00e-04]
   12      0.2591     0.9825    1.3076   0.9474  [LR → 5.00e-04]
   13      0.2561     0.9829    1.4756   0.9479  [LR → 5.00e-04]

Early stopping — no improvement for 5 epochs

Best val AUC: 0.9489 at epoch 8


In [15]:
model = GlacierCNN(dropout=0.5).to(device)
model.load_state_dict(torch.load("best_glacier_cnn.pt", weights_only=True))
model.eval()
print("Best model loaded (epoch 8)")

Best model loaded (epoch 8)


In [16]:
all_probs  = []
all_labels = []

with torch.no_grad():
    for patches, labels in val_loader:
        logits = model(patches)
        probs  = torch.sigmoid(logits)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

print(f"Val AUC: {roc_auc_score(all_labels, all_probs):.4f}")
print(f"Predictions shape: {all_probs.shape}")
print(f"Mean predicted melt probability: {all_probs.mean():.3f}")
print(f"True melt rate: {all_labels.mean():.3f}")

Val AUC: 0.9489
Predictions shape: (1774319,)
Mean predicted melt probability: 0.215
True melt rate: 0.240


In [17]:
df_r3 = pd.read_parquet("/nvme1/users/md962/glacier/Glacier Project/Merged/r3_combined.parquet")

assert len(df_r3) == len(all_probs), \
    f"Mismatch: {len(df_r3)} rows vs {len(all_probs)} predictions"

# Save just the identifying columns + predictions
df_preds = pd.DataFrame({
    'lon':       df_r3['lon'].values,
    'lat':       df_r3['lat'].values,
    'melt_label': df_r3['melt_label'].values,
    'cnn_prob':  all_probs,
    'cnn_pred':  (all_probs >= 0.5).astype(int)
})

df_preds.to_parquet("/nvme1/users/md962/glacier/Glacier Project/Predictions/cnn_predictions_r3.parquet", index=False)
print(f"Saved {len(df_preds):,} predictions")
print(df_preds.head())

Saved 1,774,319 predictions
         lon        lat  melt_label  cnn_prob  cnn_pred
0 -70.802200 -14.190822           1  0.999939         1
1 -70.802109 -14.190822           1  0.999964         1
2 -70.802109 -14.190911           1  0.999986         1
3 -70.802025 -14.190911           1  0.999984         1
4 -70.801933 -14.190911           1  0.999929         1


In [18]:
PARQUET_DIR = "/nvme1/users/md962/glacier/Glacier Project/Merged"
RESULTS_DIR = "/nvme1/users/md962/glacier/Glacier Project/Results"
REGIONS = ['r1a', 'r1b', 'r2', 'r3']

model.eval()
all_region_preds = []

for region in REGIONS:
    print(f"\nRunning inference on {region}...")
    
    # Load tensors from disk (r3 already done, others need building)
    tensors = torch.load(
        f'/nvme1/users/md962/glacier/tensors_{region}.pt', 
        weights_only=False
    )
    
    loader = GPUDataLoader(tensors, batch_size=65536, shuffle=False)
    
    probs_region  = []
    labels_region = []
    
    with torch.no_grad():
        for patches, labels in loader:
            logits = model(patches)
            probs  = torch.sigmoid(logits)
            probs_region.extend(probs.cpu().numpy())
            labels_region.extend(labels.cpu().numpy())
    
    # Load original parquet for lon/lat
    df_region = pd.read_parquet(f"{PARQUET_DIR}/{region}_combined.parquet")
    
    assert len(df_region) == len(probs_region), \
        f"Mismatch in {region}: {len(df_region)} rows vs {len(probs_region)} predictions"
    
    df_out = pd.DataFrame({
        'region':     region.upper(),
        'lon':        df_region['lon'].values,
        'lat':        df_region['lat'].values,
        'melt_label': df_region['melt_label'].values,
        'edge_distance': df_region['edge_distance'].values,
        'prob_cnn_patch3': np.array(probs_region, dtype=np.float32)
    })
    
    all_region_preds.append(df_out)
    print(f"  {region}: {len(df_out):,} pixels, "
          f"mean prob: {df_out['prob_cnn_patch3'].mean():.3f}, "
          f"true melt rate: {df_out['melt_label'].mean():.3f}")

# Combine all regions
df_cnn_preds = pd.concat(all_region_preds, ignore_index=True)
df_cnn_preds.to_parquet(f"{RESULTS_DIR}/cnn_patch3_predictions_peru.parquet", index=False)
print(f"\nSaved {len(df_cnn_preds):,} total predictions")


Running inference on r1a...
  r1a: 6,222,913 pixels, mean prob: 0.190, true melt rate: 0.139

Running inference on r1b...
  r1b: 1,028,089 pixels, mean prob: 0.343, true melt rate: 0.264

Running inference on r2...
  r2: 6,217,318 pixels, mean prob: 0.286, true melt rate: 0.227

Running inference on r3...
  r3: 1,774,319 pixels, mean prob: 0.215, true melt rate: 0.240

Saved 15,242,639 total predictions


In [19]:
# Load existing predictions
df_existing = pd.read_parquet("/nvme1/users/md962/glacier/Glacier Project/Results/predictions_full_peru.parquet")

# Load CNN predictions
df_cnn = pd.read_parquet("/nvme1/users/md962/glacier/Glacier Project/Results/cnn_patch3_predictions_peru.parquet")

print(f"Existing predictions: {len(df_existing):,} rows, columns: {df_existing.columns.tolist()}")
print(f"CNN predictions: {len(df_cnn):,} rows")

# Merge on lon/lat
df_all_preds = df_existing.merge(
    df_cnn[['lon', 'lat', 'prob_cnn_patch3']],
    on=['lon', 'lat'],
    how='left'
)

print(f"\nAfter merge: {len(df_all_preds):,} rows")
print(f"CNN null predictions: {df_all_preds['prob_cnn_patch3'].isna().sum():,}")

# Overwrite with updated file
df_all_preds.to_parquet(
    "/nvme1/users/md962/glacier/Glacier Project/Results/predictions_full_peru.parquet", 
    index=False
)
print("Saved updated predictions_full_peru.parquet")

Existing predictions: 15,242,639 rows, columns: ['region', 'lon', 'lat', 'melt_label', 'edge_distance', 'prob_easd', 'prob_pc10']
CNN predictions: 15,242,639 rows

After merge: 17,734,261 rows
CNN null predictions: 0
Saved updated predictions_full_peru.parquet


In [20]:
# Check for duplicate lon/lat in each file
print("Duplicates in existing predictions:")
print(df_existing.duplicated(subset=['lon', 'lat']).sum())

print("\nDuplicates in CNN predictions:")
print(df_cnn.duplicated(subset=['lon', 'lat']).sum())

Duplicates in existing predictions:
1245811

Duplicates in CNN predictions:
1245811


In [21]:
# Reload originals fresh (don't use the already-merged df_all_preds)
df_existing = pd.read_parquet("/nvme1/users/md962/glacier/Glacier Project/Results/predictions_full_peru.parquet")
df_cnn = pd.read_parquet("/nvme1/users/md962/glacier/Glacier Project/Results/cnn_patch3_predictions_peru.parquet")

# Deduplicate both
df_existing_dedup = df_existing.drop_duplicates(subset=['lon', 'lat'], keep='first').reset_index(drop=True)
df_cnn_dedup = df_cnn.drop_duplicates(subset=['lon', 'lat'], keep='first').reset_index(drop=True)

print(f"Existing: {len(df_existing):,} → {len(df_existing_dedup):,} after dedup")
print(f"CNN:      {len(df_cnn):,} → {len(df_cnn_dedup):,} after dedup")

# Now merge
df_all_preds = df_existing_dedup.merge(
    df_cnn_dedup[['lon', 'lat', 'prob_cnn_patch3']],
    on=['lon', 'lat'],
    how='left'
)

print(f"\nAfter merge: {len(df_all_preds):,} rows")
print(f"CNN null predictions: {df_all_preds['prob_cnn_patch3'].isna().sum():,}")

# Save
df_all_preds.to_parquet(
    "/nvme1/users/md962/glacier/Glacier Project/Results/predictions_full_peru.parquet",
    index=False
)
print("Saved updated predictions_full_peru.parquet")

Existing: 17,734,261 → 13,996,828 after dedup
CNN:      15,242,639 → 13,996,828 after dedup

After merge: 13,996,828 rows


KeyError: 'prob_cnn_patch3'

In [27]:
print(f"Rows: {len(df_all_preds):,}")
print(f"Null CNN predictions: {df_all_preds['prob_cnn_patch3'].isna().sum():,}")

df_all_preds.to_parquet(
    "/nvme1/users/md962/glacier/Glacier Project/Results/predictions_full_peru.parquet",
    index=False
)
print("Saved cleanly")

Rows: 13,996,828
Null CNN predictions: 0


Saved cleanly


In [28]:
df = pd.read_parquet("/nvme1/users/md962/glacier/Glacier Project/Results/predictions_full_peru.parquet")

MODELS = {
    'RF (EASD)':    'prob_easd',
    'MLP (AE64)':   'prob_pc10',
    'CNN (patch3)': 'prob_cnn_patch3'
}

actual_melt = df['melt_label'].values == 1
n_actual_melt = actual_melt.sum()
print(f"Total pixels:      {len(df):,}")
print(f"Actual melt pixels: {n_actual_melt:,} ({n_actual_melt/len(df)*100:.1f}%)\n")

print(f"{'Model':<20} {'Overlap %':>10} {'IoU':>8}")
print("-" * 42)

for model_name, prob_col in MODELS.items():
    probs = df[prob_col].values
    
    # Calibrated threshold: select exactly n_actual_melt pixels with highest probability
    top_indices = np.argsort(probs)[::-1][:n_actual_melt]
    predicted_melt = np.zeros(len(probs), dtype=bool)
    predicted_melt[top_indices] = True
    
    n_intersect  = (predicted_melt & actual_melt).sum()
    overlap_pct  = n_intersect / n_actual_melt * 100
    iou          = n_intersect / (predicted_melt | actual_melt).sum()
    
    print(f"{model_name:<20} {overlap_pct:>9.2f}% {iou:>8.4f}")

Total pixels:      13,996,828
Actual melt pixels: 2,628,822 (18.8%)

Model                 Overlap %      IoU
------------------------------------------
RF (EASD)                54.21%   0.3718
MLP (AE64)               67.94%   0.5144
CNN (patch3)             83.84%   0.7218


In [29]:
df_r3 = df[df['region'] == 'R3']
actual_melt_r3 = df_r3['melt_label'].values == 1
n_actual_melt_r3 = actual_melt_r3.sum()

print(f"R3 pixels: {len(df_r3):,}, melt pixels: {n_actual_melt_r3:,}\n")
print(f"{'Model':<20} {'Overlap %':>10} {'IoU':>8}")
print("-" * 42)

for model_name, prob_col in MODELS.items():
    probs = df_r3[prob_col].values
    top_indices = np.argsort(probs)[::-1][:n_actual_melt_r3]
    predicted_melt = np.zeros(len(probs), dtype=bool)
    predicted_melt[top_indices] = True
    actual_melt_r3_arr = df_r3['melt_label'].values == 1
    n_intersect = (predicted_melt & actual_melt_r3_arr).sum()
    overlap_pct = n_intersect / n_actual_melt_r3 * 100
    iou = n_intersect / (predicted_melt | actual_melt_r3_arr).sum()
    print(f"{model_name:<20} {overlap_pct:>9.2f}% {iou:>8.4f}")

R3 pixels: 528,508, melt pixels: 83,253

Model                 Overlap %      IoU
------------------------------------------
RF (EASD)                61.50%   0.4441
MLP (AE64)               76.91%   0.6249
CNN (patch3)             74.29%   0.5910
